In [2]:
pip install transformers torch datasets scikit-learn

  Using cached transformers-4.49.0-py3-none-any.whl.metadata (44 kB)
  Using cached torch-2.6.0-cp313-cp313-win_amd64.whl.metadata (28 kB)
  Using cached datasets-3.3.2-py3-none-any.whl.metadata (19 kB)
  Using cached filelock-3.17.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached regex-2024.11.6-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.21.0-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached safetensors-0.5.3-cp38-abi3-win_amd64.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached pyarrow-19.0.1-cp313-cp313-win_amd64.whl.metadata (3.4 kB)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.5.0-cp313-cp313-win_amd64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.16-py312-none-any.

In [9]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import BertTokenizer, BertModel
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim

In [10]:
df = pd.read_csv("fake_job_postings.csv")

df = df.dropna(subset=["description", "fraudulent"])

df["fraudulent"] = df["fraudulent"].astype(int)

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["description"].tolist(), df["fraudulent"].tolist(), test_size=0.2, random_state=42
)

print(f"Training samples: {len(train_texts)}, Testing samples: {len(test_texts)}")

Training samples: 14303, Testing samples: 3576


In [7]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [11]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512)

In [12]:
class JobDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = JobDataset(train_encodings, train_labels)
test_dataset = JobDataset(test_encodings, test_labels)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [13]:
class FakeJobClassifier(nn.Module):
    def __init__(self):
        super(FakeJobClassifier, self).__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(768, 1)  # BERT output dimension is 768
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output  # Use [CLS] token output
        x = self.dropout(pooled_output)
        x = self.fc(x)
        return self.sigmoid(x).squeeze()

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FakeJobClassifier().to(device)

optimizer = optim.AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.BCELoss()  # Binary Cross-Entropy Loss

def train(model, train_loader, optimizer, loss_fn, epochs=3):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].float().to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask).view(-1)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            preds = (outputs > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        accuracy = correct / total
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}, Accuracy: {accuracy:.4f}")

train(model, train_loader, optimizer, loss_fn, epochs=3)


Error while downloading from https://cdn-lfs.hf.co/bert-base-uncased/097417381d6c7230bd9e3557456d726de6e83245ec8b24f529f60198a67b203a?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27pytorch_model.bin%3B+filename%3D%22pytorch_model.bin%22%3B&response-content-type=application%2Foctet-stream&Expires=1742877861&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0Mjg3Nzg2MX19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9iZXJ0LWJhc2UtdW5jYXNlZC8wOTc0MTczODFkNmM3MjMwYmQ5ZTM1NTc0NTZkNzI2ZGU2ZTgzMjQ1ZWM4YjI0ZjUyOWY2MDE5OGE2N2IyMDNhP3Jlc3BvbnNlLWNvbnRlbnQtZGlzcG9zaXRpb249KiZyZXNwb25zZS1jb250ZW50LXR5cGU9KiJ9XX0_&Signature=a07ug-3cYVqX8i9hJM8LokylNGjnx1gHD4EMKwxZjG-F%7ExvvFf6lIxRY3oScVp0UrTBv27eGkoWY%7Ejfu1tjzbv-VHfC7KpBEKivNM%7E8D31oupSJQspyUa3-7u2IuwBTBScTCTy%7EXyZh4kOHj%7Ee0shr2DQH0v%7EBZU9ZovZWOE8D8NA1O1Jz40mmj4do8oYzVks8cChBdNeRrPwZPh4ynFX0Cxhv7v9iSgHJQBXIm8vS42JV1AUFMJkK42sXyYWCOZnAbgMJVlYrJAFkeVz49NEiPz-13-fNb-fyv7iWYOx3eZS5yb04pqLGEVJfc

ConnectionError: (MaxRetryError('HTTPSConnectionPool(host=\'cdn-lfs.hf.co\', port=443): Max retries exceeded with url: /bert-base-uncased/097417381d6c7230bd9e3557456d726de6e83245ec8b24f529f60198a67b203a?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27pytorch_model.bin%3B+filename%3D%22pytorch_model.bin%22%3B&response-content-type=application%2Foctet-stream&Expires=1742877861&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0Mjg3Nzg2MX19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9iZXJ0LWJhc2UtdW5jYXNlZC8wOTc0MTczODFkNmM3MjMwYmQ5ZTM1NTc0NTZkNzI2ZGU2ZTgzMjQ1ZWM4YjI0ZjUyOWY2MDE5OGE2N2IyMDNhP3Jlc3BvbnNlLWNvbnRlbnQtZGlzcG9zaXRpb249KiZyZXNwb25zZS1jb250ZW50LXR5cGU9KiJ9XX0_&Signature=a07ug-3cYVqX8i9hJM8LokylNGjnx1gHD4EMKwxZjG-F~xvvFf6lIxRY3oScVp0UrTBv27eGkoWY~jfu1tjzbv-VHfC7KpBEKivNM~8D31oupSJQspyUa3-7u2IuwBTBScTCTy~XyZh4kOHj~e0shr2DQH0v~BZU9ZovZWOE8D8NA1O1Jz40mmj4do8oYzVks8cChBdNeRrPwZPh4ynFX0Cxhv7v9iSgHJQBXIm8vS42JV1AUFMJkK42sXyYWCOZnA

In [ ]:
def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].float().to(device)

            outputs = model(input_ids, attention_mask).view(-1)
            preds = (outputs > 0.5).float()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total
    print(f"Test Accuracy: {accuracy:.4f}")
    print("Classification Report:")
    print(classification_report(all_labels, all_preds))

evaluate(model, test_loader)


In [ ]:
def predict_job_posting(model, tokenizer, text):
    model.eval()
    encoding = tokenizer(text, truncation=True, padding=True, max_length=512, return_tensors="pt")
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        output = model(input_ids, attention_mask).item()

    prediction = "FAKE JOB ❌" if output > 0.5 else "REAL JOB ✅"
    print(f"Prediction: {prediction} (Confidence: {output:.4f})")

# Example usage
job_posting_text = "Remote data entry job, no experience needed, earn $5000 per week. Just send us your resume and processing fee."
predict_job_posting(model, tokenizer, job_posting_text)
